In [ ]:
# Import Required Libraries 
from apscheduler.schedulers.background import BackgroundScheduler
from sqlalchemy import create_engine, inspect
from datetime import datetime
import io
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
#     def read_data(): 
#     file_path = input("Enter your dataset path ")
#     cleaned_path = file_path.strip('"\'')
#     safe_path = Path(cleaned_path)
#     data = pd.read_csv(safe_path)
#     return data
    

In [3]:
#read_data()

In [4]:
# import pandas as pd

# # Use the 'r' prefix before the string to handle Windows backslashes safely
# path = r"C:\Users\Mosta\Downloads\8_4_competition_results.csv"
# df = pd.read_csv(path)

In [ ]:
data = None

In [ ]:
def user_choice():
    print("\nHow would you like to calculate your ROP and check stock levels?")
    print("  1. Read from dataset and SQL database")
    print("  2. ROP calculator(Manual)")
    choice = input("\nEnter 1 or 2: ").strip()
    return choice
    

In [ ]:
def manual_rop_calculator():
    config = {
        "current_stock" : int(input("Enter current stock level: ")),
        "avg_sales" : int(input("Enter average sales per day: ")),
        "std_dev_sales" : int(input("Enter standard deviation of sales per day: ")),
        "lead_time" : int(input("Enter lead time in days: ")),
    }
    # FIX: Removed trailing comma that accidentally cast safety_stock into a tuple
    config["safety_stock"] = 1.65 * config["std_dev_sales"] * np.sqrt(config["lead_time"])
    config["rop"] = (config["avg_sales"] * config["lead_time"]) + config["safety_stock"]
    
    if config["current_stock"] <= config["rop"]:
        print(f"⚠️  REORDER ALERT")
        print(f"    Current stock : {config['current_stock']} units")
        print(f"    Reorder point : {config['rop']:.1f} units")
        print(f"    → Place a new order now")
    else:
        buffer = config["current_stock"] - config["rop"]
        print(f"✅  Stock is OK")
        print(f"    Current stock : {config['current_stock']} units")
        print(f"    Reorder point : {config['rop']:.1f} units")
        print(f"    → {buffer:.1f} units above reorder point")

In [ ]:
def read_data_automatically():
    # 1. Create a visual upload button
    uploader = widgets.FileUpload(
        accept=".csv",  # Only allow CSV files
        multiple=False,  # Single file only
    )
    display(uploader)

    # 2. Internal function to process the file once clicked
    def on_upload_change(change):
        if not uploader.value:
            return

        # Grab the file data directly from memory
        uploaded_file = uploader.value[0]
        file_content = uploaded_file["content"]

        # Read it directly into pandas (no paths needed!)
        global data
        data = pd.read_csv(io.BytesIO(file_content))
        print(f"Successfully loaded: {uploaded_file['name']}")

    # Watch for the user to select a file
    uploader.observe(on_upload_change, names="value")
    
    

# Run this cell to show the button
# read_data_automatically()

In [ ]:
def find_column(df, user_input):
    # ENHANCEMENT: Also handles underscores (.replace('_', ' ')) so typing 'inventory_level' matches 'Inventory Level'
    matched = [col for col in df.columns if col.lower().replace('_', ' ').strip() == user_input.lower().replace('_', ' ').strip()]
    if matched:
        return matched[0]   # returns the real column name as it exists in the df
    raise ValueError(f"Column '{user_input}' not found. Available: {list(df.columns)}")

In [ ]:
def find_date_column(df):
    synonyms = ['date', 'datetime', 'timestamp', 'time', 'day', 'period']
    for col in df.columns:
        col_lower = col.lower().strip()
        if col_lower in synonyms or any(s in col_lower for s in synonyms):
            return col
    return None

In [ ]:
def get_product(data):
    config = {
        "product_column": find_column(data, input("Enter product column name: ")),
        "product_name":   input("Enter the product name: ").strip(),
        "sales_column":   find_column(data, input("Enter sales column name: ")),
        "lead_time":      int(input("Lead time (days): ")),
        "restock_days":   int(input("How many days between restocks (e.g. 30, 0 if unknown): ")),
        "review_period":  int(input("Recalculate ROP every how many days? (90 recommended): ")),
    }
    config["targeted_sales"] = data.loc[
        data[config["product_column"]] == config["product_name"]
    ].copy()
    
    return (
        config["targeted_sales"],
        config["sales_column"],
        config["lead_time"],
        config["restock_days"],
        config["review_period"],
    )

In [ ]:
def get_db_data():
    dataconfig = {
        "Sales_table":  input("Enter sales table name: ").strip(),
        "sales_column": input("Enter sales column name: ").strip(),
        "lead_time":    int(input("Lead time (days): ")),
        "restock_days": int(input("How many days between restocks (e.g. 30, 0 if unknown): ")),
        "review_period":int(input("Recalculate ROP every how many days? (90 recommended): ")),
    }
    return dataconfig

In [ ]:
def sql_connection(dataconfig):
    dbconfig = {
        "db_type":       input("Enter database type (mysql / postgresql / sqlite): ").strip().lower(),
        "Host":          input("Localhost or IP address: ").strip(),
        "Port":          input("Port (3306 for MySQL, 5432 for PostgreSQL): ").strip(),
        "Username":      input("Database username: ").strip(),
        "Password":      input("Database password: ").strip(),
        "Database_name": input("Database name: ").strip(),
    }
 
    drivers = {
        "mysql":      "mysql+pymysql",
        "postgresql": "postgresql+psycopg2",
        "sqlite":     "sqlite",
    }
    db_type = dbconfig["db_type"]
    driver  = drivers.get(db_type, db_type)
 
    engine = create_engine(
        f"{driver}://{dbconfig['Username']}:{dbconfig['Password']}"
        f"@{dbconfig['Host']}:{dbconfig['Port']}/{dbconfig['Database_name']}"
    )
 
    # Show real column names from the database before asking user to type them
    inspector = inspect(engine)
    columns   = [col['name'] for col in inspector.get_columns(dataconfig['Sales_table'])]
    print(f"\n  Available columns: {columns}")
 
    # Ask for product column AFTER user sees the list
    dataconfig['product_column'] = input("\n  Enter exact product column name: ").strip()
    if dataconfig['product_column'] not in columns:
        raise ValueError(
            f"Column '{dataconfig['product_column']}' not found. Choose from: {columns}"
        )
 
    # Ask for product name AFTER validation passes
    dataconfig['product_name'] = input("  Enter product name: ").strip()
 
    # Query the database
    targeted_sales = pd.read_sql(
        f"SELECT * FROM `{dataconfig['Sales_table']}` "
        f"WHERE `{dataconfig['product_column']}` = '{dataconfig['product_name']}'",
        engine
    )
    return targeted_sales

In [ ]:
def data_sentizer(targeted_sales, sales_column):
    
    def duplicates_sentizer(targeted_sales):
        if targeted_sales.duplicated().sum() != 0:  
           # FIX: Changed 'df' to 'targeted_sales' to resolve NameError namespace issue
           targeted_sales = targeted_sales.drop_duplicates().copy()
        return targeted_sales
    
    def outliers_sentizer(ts_df):
        q1 = pd.Series(sorted(ts_df[sales_column])).quantile(0.25)
        q3 = pd.Series(sorted(ts_df[sales_column])).quantile(0.75)
        IQR = q3 - q1
        upper_fence = q3 + 1.5 * IQR
        lower_fence = max(0, q1 - 1.5 * IQR)
        # FIX: Working safely and directly on the scoped variable mapping
        ts_df.loc[:, sales_column] = ts_df[sales_column].clip(lower_fence, upper_fence)
        return ts_df
    
    def nulls_sentizer(targeted_sales):   
        if targeted_sales[sales_column].isna().sum() != 0: 
            targeted_sales.loc[:, sales_column] = targeted_sales[sales_column].fillna(targeted_sales[sales_column].mean())       
        return targeted_sales

    targeted_sales = duplicates_sentizer(targeted_sales)
    targeted_sales = nulls_sentizer(targeted_sales)
    targeted_sales = outliers_sentizer(targeted_sales)
    return targeted_sales

In [ ]:
def ROP(targeted_sales, sales_column, lead_time):
    # FIX: Balanced signature layout to accept lead_time and return 3 items as main() expects
    avg_sales = targeted_sales[sales_column].mean()
    std_dev_sales = targeted_sales[sales_column].std()
    
    # Handle single row edge-case scenario where std evaluates to NaN
    if pd.isna(std_dev_sales):
        std_dev_sales = 0.0

    z_score = 1.65 
    safety_stock = z_score * std_dev_sales * np.sqrt(lead_time)
    rop = (avg_sales * lead_time) + safety_stock
    
    return rop, avg_sales, std_dev_sales

In [ ]:
def order_quantity(avg_sales, rop, restock_days):
    print("\nWould you like to calculate your order quantity?")
    choice = input("1 for yes, 2 for no: ").strip()
    if choice == "1":
        print("\n  1. Based on a specific quantity (units)")
        print("  2. Based on desired stock duration (days)")
        choice = input("\n  Enter 1 or 2: ").strip()
        if choice == "1":
            qty  = int(input("\n  Enter the desired order quantity: "))
            days = (qty - rop) / avg_sales if avg_sales != 0 else 0
            print(f"  → {qty} units will last approximately {days:.1f} days")
        else:
            days = restock_days
            qty  = rop + (avg_sales * days)
            print(f"  → Recommended order quantity for {days} days: {qty:.1f} units")

In [ ]:
def stock_alert(rop, targeted_sales, avg_sales=None, std_dev_sales=None):
    # FIX: Added optional variables to signature so it safely processes the 4 arguments passed from main()
    print("\nHow would you like to provide the current stock level?")
    print("  1. Read from dataset (if your dataset has an inventory column)")
    print("  2. Enter manually")
    choice = input("\nEnter 1 or 2: ").strip()
    if choice == "1":
        date_col = find_date_column(targeted_sales)
        if date_col:
            latest_row = targeted_sales.sort_values(date_col).iloc[-1]
        else:
            print("  ℹ️  No date column found — using last row as most recent")
            latest_row = targeted_sales.iloc[-1]
        inventory_col = find_column(targeted_sales, input("Enter inventory/stock column name: "))
        current_stock = latest_row[inventory_col]
        print(f"  → Latest inventory level from dataset: {current_stock} units")
    else:
        current_stock = int(input("Enter current stock level: "))
    
    print()
    if current_stock <= rop:
        print(f"⚠️  REORDER ALERT")
        print(f"    Current stock : {current_stock} units")
        print(f"    Reorder point : {rop:.1f} units")
        print(f"    → Place a new order now")
    else:
        buffer = current_stock - rop
        print(f"✅  Stock is OK")
        print(f"    Current stock : {current_stock} units")
        print(f"    Reorder point : {rop:.1f} units")
        print(f"    → {buffer:.1f} units above reorder point")

In [ ]:
def daily_stock_check(targeted_sales, sales_column, rop, avg_sales):
    print(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M')}] 🔄 Daily stock check...")
    try:
        date_col  = find_date_column(targeted_sales)
        latest_row = (
            targeted_sales.sort_values(date_col).iloc[-1]
            if date_col else targeted_sales.iloc[-1]
        )
        current_stock = latest_row[sales_column]
        if current_stock <= rop:
            print(f"  ⚠️  REORDER ALERT")
            print(f"      Current stock : {current_stock} units")
            print(f"      Reorder point : {rop:.1f} units")
            print(f"      → Place a new order now")
        else:
            print(f"  ✅  Stock OK — {current_stock - rop:.1f} units above ROP")
    except Exception as e:
        print(f"  ❌ Daily check failed: {e}")

def recalculate_rop(targeted_sales, sales_column, lead_time):
    print(f"\n[{datetime.now().strftime('%Y-%m-%d %H:%M')}] 📊 Recalculating ROP...")
    try:
        cleaned               = data_sentizer(targeted_sales, sales_column)
        new_rop, new_avg, new_std = ROP(cleaned, sales_column, lead_time)
        print(f"  ✅  New ROP : {new_rop:.1f} units")
        print(f"      Avg     : {new_avg:.1f} | Std dev: {new_std:.1f}")
        return new_rop
    except Exception as e:
        print(f"  ❌ Recalculation failed: {e}")
 
 
def start_scheduler(targeted_sales, sales_column, rop, avg_sales,
                    std_dev_sales, lead_time, review_period):
    review_weeks = max(1, review_period // 7)
    scheduler    = BackgroundScheduler()
 
    scheduler.add_job(
        lambda: daily_stock_check(targeted_sales, sales_column, rop, avg_sales),
        trigger='cron', hour=8, id='daily_check'
    )
    scheduler.add_job(
        lambda: recalculate_rop(targeted_sales, sales_column, lead_time),
        trigger='interval', weeks=review_weeks, id='rop_recalc'
    )
    scheduler.start()
    print(f"✅  Scheduler started")
    print(f"    Daily stock check  : every day at 8:00 AM")
    print(f"    ROP recalculation  : every {review_weeks} weeks")
    return scheduler

In [18]:
data = read_data_automatically()

FileUpload(value=(), accept='.csv', description='Upload')

In [20]:
data.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


In [ ]:
def main():
    choice = user_choice()
 
    if choice == "1":
        print("\nWould you like to read data from a SQL database or a CSV file?")
        source = input("Enter 1 for SQL database or 2 for CSV file: ").strip()
 
        if source == "1":
            dataconfig     = get_db_data()
            targeted_sales = sql_connection(dataconfig)
            sales_column   = dataconfig["sales_column"]
            lead_time      = dataconfig["lead_time"]
            restock_days   = dataconfig["restock_days"]
            review_period  = dataconfig["review_period"]
        else:
            if data is None:
                print("❌ No data loaded. Please execute read_data_automatically() first.")
                return
            targeted_sales, sales_column, lead_time, restock_days, review_period = get_product(data)
 
        # Clean data and execute target logic pipeline
        targeted_sales = data_sentizer(targeted_sales, sales_column)
        rop, avg_sales, std_dev_sales = ROP(targeted_sales, sales_column, lead_time)
        stock_alert(rop, targeted_sales, avg_sales, std_dev_sales)
        order_quantity(avg_sales, rop, restock_days)
        start_scheduler(targeted_sales, sales_column, rop, avg_sales, std_dev_sales, lead_time, review_period)
    else:
        manual_rop_calculator()

In [ ]:
main()


How would you like to calculate your ROP and check stock levels?
  1. Read from dataset and SQL database
  2. ROP calculator(Manual)

Would you like to read data from a SQL database or a CSV file?


OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'ys'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)